In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Downloading Necessary Libraries & Packages

In [ ]:
!pip install open3d
!apt install octave

## Importing Necessary Libraries, Packages and Functions

In [4]:
import open3d as o3d
import numpy as np
import nibabel as nib
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import map_coordinates
import matplotlib.pyplot as plt

from cehe_algorithm import get_density_map, cehe, inverted_average_fn
from test_aug_mat import test_edge_aug_matrix_1, test_edge_aug_matrix_2

## Some Utility Functions:
These functions should be implemented in AOMT and the algorithm implemented here should be supplied by vertices coordinates of the mesh, faces of the mesh, and density of each vertex.

In [5]:
def build_density_map(file_path):
  img = nib.load(file_path)
  img_arr = img.get_fdata()
  normalized_img_arr = (img_arr - np.mean(img_arr)) / np.std(img_arr)
  enhanced_img_arr = CEHE(normalized_img_arr, inverted_average_fn, 3, 65536)
  enhanced_image = nib.Nifti1Image(enhanced_img_arr, affine=np.eye(4))
  return get_density_map(enhanced_image, 1)

In [6]:
def read_off_file(file_path):
  mesh = o3d.io.read_triangle_mesh(file_path)
  vertices = np.asarray(mesh.vertices)
  faces = np.asarray(mesh.triangles)
  return vertices, np.sort(faces, axis=1)

In [7]:
d_map = build_density_map("/content/drive/MyDrive/Graduation Project/Datasets/flair samples/BraTS2021_00000_flair.nii.gz")

In [8]:
vertices, faces = read_off_file('/content/BraTS2021_Training_00000_flair.off')

## A Function to compute vertices density
It performs Trilinear Interpolation on each vertex coordinate to get its density from the image density map.

In [9]:
get_vertices_density = lambda d_map, vertices: map_coordinates(d_map, vertices.T, order=1, mode='nearest')

## Building Edge Augmented Matrix

Edge Augmented Matrix is a **(#edges x 8) matrix**, its columns are as follows: </br>
1. **Columns {0, 1}:** Indices of the first and second vertices of the edge.
2. **Columns {2, 3}:** Indices of the 2 vertices such that if each is combined with the edge, a triangle in the mesh is formed.
3. **Columns {4, 5}:** Areas of the Two Triangles formed by the vertices in the previous 2 columns and the edge respectively.
4. **Columns {6, 7}:** Densities of the Two Triangles formed by the vertices in the previous 2 columns and the edge respectively.

### a Function to build Adjacency Matrix for the Mesh

In [10]:
def build_adj_matrix(mesh_faces, num_vertices):
    adj_matrix = np.zeros((num_vertices, num_vertices), dtype=bool)

    def fill_adj_matrix(face):
      v1, v2, v3 = face[0], face[1], face[2]
      adj_matrix[v1, v2], adj_matrix[v1, v3], adj_matrix[v2, v3] = True, True, True
      adj_matrix[v2, v1], adj_matrix[v3, v1], adj_matrix[v3, v2] = True, True, True

    np.apply_along_axis(fill_adj_matrix, axis=1, arr=mesh_faces)
    return adj_matrix

### a Function to build the first 4 comlums in Edge Augmented Matrix

In [11]:
def build_edge_aug_matrix_f4(adj_matrix, mesh_faces):
  edge_aug_matrix = []
  row_indices, col_indices = np.where(adj_matrix)
  edges = np.vstack([row_indices, col_indices]).T
  edges = edges[edges[:, 0] < edges[:, 1]]

  def get_triangle_vertices(edge):
    common_vertices = adj_matrix[edge[0]] & adj_matrix[edge[1]]
    if np.sum(common_vertices) > 1: # we have two triangles sharing this edge
      possible_face_heads = np.where(common_vertices)[0]
      e0_repeat = np.full_like(possible_face_heads, edge[0])
      e1_repeat = np.full_like(possible_face_heads, edge[1])

      possible_faces = np.column_stack((e0_repeat, e1_repeat, possible_face_heads))
      possible_faces = np.sort(possible_faces, axis=1)
      face_indices = np.where(np.all(possible_faces[:, None, :] == mesh_faces, axis=-1))[0]

      if(len(face_indices) > 1):
        x = np.array(np.hstack([edge, possible_face_heads[face_indices]])[:])
        edge_aug_matrix.append(x)

  np.apply_along_axis(get_triangle_vertices, axis=1, arr=edges)
  return np.array(edge_aug_matrix)

### a Function to compute columns 4 and 5 in Edge Augmented Matrix

In [12]:
compute_face_area = lambda v1, v2, v3: 0.5 * np.linalg.norm(np.cross(v2 - v1, v3 - v1))

In [13]:
def build_edge_aug_matrix_4_5(edge_aug_matrix, vertices):
  def compute_entry_4_5(row):
    v1, v2 = vertices[int(row[0])], vertices[int(row[1])]
    v3, v4 = vertices[int(row[2])], vertices[int(row[3])]
    row[4:6] = np.array([compute_face_area(v1, v2, v3), compute_face_area(v1, v2, v4)])

  np.apply_along_axis(compute_entry_4_5, axis=1, arr=edge_aug_matrix)
  return edge_aug_matrix[:, 4:6]

### a Function to compute columns 6 and 7 in Edge Augmented Matrix

In [14]:
def build_edge_aug_matrix_6_7(edge_aug_matrix, vertices_densities):
  def compute_entry_6_7(row):
    d_v1, d_v2 = vertices_densities[int(row[0])], vertices_densities[int(row[1])]
    d_v3, d_v4 = vertices_densities[int(row[2])], vertices_densities[int(row[3])]
    row[6:8] = np.array([(d_v1 + d_v2 + d_v3) / 3, (d_v1 + d_v2 + d_v4) / 3])

  np.apply_along_axis(compute_entry_6_7, axis=1, arr=edge_aug_matrix)
  return edge_aug_matrix[:, 6:8]

### a Facade Function to build Edge Augmented Matrix

In [15]:
def build_edge_aug_matrix(adj_matrix, vertices, faces, vertices_densities):
  edge_aug_matrix_mini = build_edge_aug_matrix_f4(adj_matrix, faces)
  edge_aug_matrix = np.zeros((edge_aug_matrix_mini.shape[0], 8))
  edge_aug_matrix[:, :4] = edge_aug_matrix_mini.reshape((edge_aug_matrix_mini.shape[0], 4))
  edge_aug_matrix[:, 4:6] = build_edge_aug_matrix_4_5(edge_aug_matrix, vertices)
  edge_aug_matrix[:, 6:8] = build_edge_aug_matrix_6_7(edge_aug_matrix, vertices_densities)
  return edge_aug_matrix

### Compute Spherical Conformal Parameterization

In [16]:
def compute_spherical_conformal_parameterization():
  # note : paths have to be changed after uploading the matlab code on github
  %cd /content/drive/MyDrive/LinearSphericalConformalMap/
  !octave -W demo.m /content/BraTS2021_Training_00000_flair.off
  %cd /content
  g = []
  with open('/content/drive/MyDrive/LinearSphericalConformalMap/spherical_conformal_map_output.txt', 'r') as file:
    num_vertices = file.readline()
    # Extract the number of vertices from the first line
    num_vertices = int(num_vertices.strip())

    print(f"Number of vertices: {num_vertices}")
    for line in file:
        g.append(line.split())
  return np.array(g, dtype=np.float64)

In [17]:
# stereo_graphic_projection = lambda v : v[0] / (1 - v[2]) + v[1] / (1 - v[2]) * 1j
ε = 1e-6

stereo_graphic_projection = lambda v: v[0] / (1 - v[2]) + v[1] / (1 - v[2]) * 1j if v[2] != 1 else 1/ε + 1j/ε

def inv_stereo_graphic_projection(complex_num):
  u = complex_num.real
  v = complex_num.imag
  norm = abs(complex_num)
  return np.array([2 * u / (norm + 1), 2 * v / (norm + 1), (norm - 1) / (norm + 1)])

def cotan_α(vi, vj, vk):
  v1, v2 = vi - vk, vj - vk
  dot_product = np.dot(v1, v2)
  cos = dot_product / ((np.linalg.norm(v1) + ε) * (np.linalg.norm(v2) + ε))
  angle = np.arccos(np.clip(cos, -1.0, 1.0))
  tan = np.tan(angle)
  if tan == 0:
    tan = ε
  return 1 / tan

def stretch_factor(ρ, area, f, i, j, k):
   f_area = compute_face_area(f[i], f[j], f[k])
  #  print('inside stretch_factor func', f_area, ρ, area, ρ * area / f_area)
   if f_area == 0:
    return ρ * area / ε
   return ρ * area / f_area

dirichlet_energy = lambda L, f: np.trace(np.dot(np.dot(f.T, L), f)) / 2.0

In [18]:
def compute_laplacian_matrix(num_vertices, edge_aug_matrix, f, ε=1e-6):
  L = np.zeros((num_vertices, num_vertices))
  count = 0
  count_pos = 0
  def compute_Lij(row):
    nonlocal count
    nonlocal count_pos
    i, j, k, l, area1, area2, ρ1, ρ2 = row
    i, j, k, l = map(int, [i, j, k, l])
    # print(f[i], f[j], f[k], f[l])
    # print('cotan', cotan_α(*f[[i, j, k]]), cotan_α(*f[[i, j, l]]))
    # print('stretch_factor', stretch_factor(ρ1, area1, f, i, j, k), stretch_factor(ρ2, area2, f, i, j, l))
    L[i, j] = cotan_α(*f[[i, j, k]]) / (stretch_factor(ρ1, area1, f, i, j, k) + ε) + cotan_α(*f[[i, j, l]]) / (stretch_factor(ρ2, area2, f, i, j, l) + ε)
    L[i, j] *= -0.5
    if i != j and L[i, j] > 0:
      L[i, j] = 0
    L[j, i] = L[i, j]
    # if cotan_α(*f[[i, j, k]]) + cotan_α(*f[[i, j, l]]) < 0:
    #   print(f'L{i},{j} {L[i, j]}')
    #   count += 1
    # if L[i, j] > 0 and i != j:
    #   count_pos += 1
    # print(f'L{i},{j} {L[i, j]} L{j},{i} {L[j, i]}')
  np.apply_along_axis(compute_Lij, axis=1, arr=edge_aug_matrix)
  # print("Count of Negative sum of cotans", count)
  # print("Num of positives not in diagonal", count_pos)
  return L + -1 * np.diag(np.sum(L, axis=1))

In [19]:
def shuffle_L_h(L, h, r=1.2):
  norms = abs(h)
  shuffled_indices = np.argsort(norms, kind='mergesort')
  shuffled_L = L[shuffled_indices]
  shuffled_h = h[shuffled_indices]
  reverse_mapping = np.argsort(shuffled_indices)
  return shuffled_L, shuffled_h, np.argmax(abs(shuffled_h) >= r), reverse_mapping

In [20]:
def inv_stereo_graphic_projection_all_points(h):
  vectorized = np.vectorize(inv_stereo_graphic_projection, signature='()->(n)')
  return vectorized(h)

In [21]:
def is_positive_semi_definite(matrix):
    # Check if the matrix is square
    if matrix.shape[0] != matrix.shape[1]:
        return False

    # Check if the matrix is symmetric
    if not np.allclose(matrix, matrix.T):
        raise Exception("matrix not symmetric")
        return False

    # Check if all eigenvalues are non-negative
    eigenvalues, _ = np.linalg.eigh(matrix)

    return np.all(eigenvalues >= 0)

In [22]:
def check_laplacian(L):
  indices = np.where(L > 0)
  print(np.count_nonzero(indices[0] != indices[1]))
  return np.all(indices[0] == indices[1])

In [23]:
num_vertices = vertices.shape[0]
vertices_densities = get_vertices_density(d_map, vertices)
adj_matrix = build_adj_matrix(faces, vertices.shape[0])
edge_aug_matrix = build_edge_aug_matrix(adj_matrix, vertices, faces, vertices_densities)
L = compute_laplacian_matrix(num_vertices, edge_aug_matrix, vertices)

In [24]:
# mask = np.diag(np.full((num_vertices), True))
# mask2 = L > 0
# print(mask2)
# print(np.count_nonzero(mask & mask2))
print(check_laplacian(L))

0
True


## ASEM Algorithm Main Function

In [33]:
def ASEM(mesh_vertices, mesh_faces, image_density_map, g=None, ε=1e-6, r=1.2):
  num_vertices = mesh_vertices.shape[0]
  vertices_densities = get_vertices_density(image_density_map, vertices)
  adj_matrix = build_adj_matrix(mesh_faces, mesh_vertices.shape[0])
  edge_aug_matrix = build_edge_aug_matrix(adj_matrix, mesh_vertices, mesh_faces, vertices_densities)
  if g is None:
    g = compute_spherical_conformal_parameterization()
    save_mesh(g, faces, 'conformal_mesh.off')
  print('g shape', g.shape)
  h = np.apply_along_axis(stereo_graphic_projection, axis=1, arr=g)
  print('h shape', h.shape)
  L = compute_laplacian_matrix(num_vertices, edge_aug_matrix, g)
  print('laplacian has positive values only on diagonal elements', check_laplacian(L))
  if is_positive_semi_definite(L):
    print("positive_semi_definite")
  else:
    print("not positive_semi_definite")
  dir_energy = dirichlet_energy(L, g)
  print('dirichlet energy =', dir_energy)
  δ = np.inf
  min_energy = np.inf;
  while δ > ε:
    # h = h / abs(h) ** 2
    h = np.divide(h, abs(h) ** 2, out=np.full_like(h, 1/ε + 1j/ε), where=(abs(h) != 0))

    print("h values: ", np.median(abs(h)), np.mean(abs(h)), np.min(abs(h)), np.max(abs(h)));
    print("h < median: ", np.count_nonzero(abs(h) < np.median(abs(h))))

    tempL, h, b_first_index, mapping = shuffle_L_h(L, h, r)
    Lii, Lib = tempL[:b_first_index, :b_first_index], tempL[:b_first_index, b_first_index:]

    hi, hb = h[:b_first_index], h[b_first_index:]
    print(Lii.shape, Lib.shape, hi.shape, hb.shape)
    # hi = np.linalg.solve(Lii, -np.dot(Lib, hb))
    hi, *_ = np.linalg.lstsq(Lii, -np.dot(Lib, hb), rcond=None)
    h[:b_first_index] = hi
    h = h[mapping]
    print('first index', b_first_index)
    g = inv_stereo_graphic_projection_all_points(h)
    nan_mask = np.isnan(g) | np.isinf(g)
    print('nan_mask', np.count_nonzero(nan_mask.astype(int)))
    new_dir_energy = dirichlet_energy(L, g)
    print('dirichlet energy =', new_dir_energy)
    δ = abs(dir_energy - new_dir_energy)
    print('δ =', δ)
    dir_energy = new_dir_energy
    L = compute_laplacian_matrix(num_vertices, edge_aug_matrix, g)
    if is_positive_semi_definite(L):
      print("positive_semi_definite")
    else:
      print("not positive_semi_definite")

    if new_dir_energy < min_energy:
      min_energy = new_dir_energy
      save_mesh(g, faces, 'asem_mesh.off')
  return g

In [34]:
g = ASEM(vertices, faces, d_map)

/content/drive/MyDrive/LinearSphericalConformalMap
double
double
    spherical_conformal_map at line 59 column 3
    demo at line 67 column 5

    linear_beltrami_solver at line 47 column 5
    spherical_conformal_map at line 112 column 5
    demo at line 67 column 5

Spherical conformal map written to spherical_conformal_map_output.txt
/content
Number of vertices: 4441
g shape (4441, 3)
h shape (4441,)
0
laplacian has positive values only on diagonal elements True
not positive_semi_definite
dirichlet energy = 0.008556114834052704
h values:  1.2860158465674878 1.5345505851298318 7.071067811865477e-07 140.47603807727475
h < median:  2220
(1986, 1986) (1986, 2455) (1986,) (2455,)
hi after update (1986,)
h after assigning hi (4441,)
h after in original order (4441,)
first index 1986
in loop h shape (4441,)
[-3.87456191e-01-1.28464231e+00j  1.56181165e+00+7.18948817e-01j
  1.45589119e+00+1.53699133e+00j ...  1.33745875e+00+1.18591000e+00j
  1.30920401e+00+1.17780254e+00j  1.21977983e-11-2.

KeyboardInterrupt: ignored

In [29]:
def save_mesh(vertices, faces, file_name):
  mesh = o3d.geometry.TriangleMesh()
  mesh.vertices = o3d.utility.Vector3dVector(vertices)
  mesh.triangles = o3d.utility.Vector3iVector(faces)

  # Save the mesh to an OFF file
  o3d.io.write_triangle_mesh(file_name, mesh)

## Tests:

In [ ]:
adj_matrix = build_adj_matrix(faces, vertices.shape[0])
edge_aug_matrix = build_edge_aug_matrix(adj_matrix, vertices, faces, get_vertices_density(d_map, vertices))

In [ ]:
test_edge_aug_matrix_2(edge_aug_matrix[78], vertices, get_vertices_density(d_map, vertices))

Areas and Densities are Correct
 Success!


### 1. Testing if first columns of Edge Augmented Matrix formed correctly

In [ ]:
test_edge_aug_matrix_1(build_adj_matrix, build_edge_aug_matrix_f4)

[[False  True False False False  True  True]
 [ True False  True False False False  True]
 [False  True False  True False False  True]
 [False False  True False  True  True  True]
 [False False False  True False  True  True]
 [ True False False  True  True False  True]
 [ True  True  True  True  True  True False]]
Adjacency Matrix is correct
[[0 6 1 5]
 [1 6 0 2]
 [2 6 1 3]
 [3 4 5 6]
 [3 6 2 4]
 [4 5 3 6]
 [4 6 3 5]
 [5 6 0 4]]
First 4 Columns of Edge Augmented Matrix are correct
 Success!
